# **The Holy Grail of Industrial CoPilot**
Task details and data: [Github](https://github.com/Bosch-ConnectedExperience-2024/BD_HackChallenge)

---


- Team: **DocCrawlers**
- BCX24 Berlin 26-28 Feb



# 1. Read\load the technical PDF texts and extract them

In [ ]:
#!pip install PyMuPDF

In [ ]:
import requests
import fitz  # PyMuPDF
from transformers import AutoTokenizer, AutoModelForQuestionAnswering
import re

# Step 1: Read and extract textual content from the given PDF
#pdf_url = "https://raw.githubusercontent.com/Bosch-ConnectedExperience-2024/Data/002_Sicherheitshinweis%20Inspektion%20(DM0009AU02)_V1.pdf"
pdf_url = "/content/002_Sicherheitshinweis Inspektion (DM0009AU02)_V1.pdf"
response = requests.get(pdf_url)
with open("pdf_file.pdf", "wb") as pdf_file:
    pdf_file.write(response.content)

response

<Response [400]>

In [ ]:
import fitz  # PyMuPDF
from transformers import pipeline
import os

# Step 1: Read and extract textual content from the given PDF
#pdf_path = "/content/002_Sicherheitshinweis_Inspektion_(DM0009AU02)_V1.pdf"
pdf_path = "/content/010_KRC T Instandsetzung Mechanik (DM0009AWQH)_V1.pdf"

def extract_text_from_pdf(pdf_path):
    text_by_page = {}
    with fitz.open(pdf_path) as doc:
        for page_num in range(len(doc)):
            page = doc.load_page(page_num)
            text = page.get_text()
            text_by_page[page_num + 1] = text
    return text_by_page

text_by_page = extract_text_from_pdf(pdf_path)

# Step 2: Apply LLM for the input query prompt
qa_pipeline = pipeline("question-answering", model="deepset/roberta-base-squad2", tokenizer="deepset/roberta-base-squad2")

query = "Wie wechsele ich das Netzteil?"

def answer_question(question, context):
    result = qa_pipeline(question=question, context=context)
    return result["answer"]

# Search for the query in the extracted text
answer = None
for page_num, text in text_by_page.items():
    if query.lower() in text.lower():
        answer = answer_question(query, text)
        print("Answer found in Page", page_num, ":", answer)
        break

if not answer:
    print("Answer not found in the provided PDF.")

# Step 3: Gather additional information from referenced PDF files (if available)
# For demonstration purposes, let's assume we have another PDF file in the same directory
additional_pdf_path = "/content/another_pdf.pdf"

if os.path.exists(additional_pdf_path):
    additional_text_by_page = extract_text_from_pdf(additional_pdf_path)
    for page_num, text in additional_text_by_page.items():
        if query.lower() in text.lower():
            additional_answer = answer_question(query, text)
            print("Additional information found in Page", page_num, ":", additional_answer)
            # You can continue processing additional references as needed
            break
    else:
        print("Additional information not found in the referenced PDF.")
else:
    print("No additional PDF file found.")


Answer not found in the provided PDF.
No additional PDF file found.


# 2. Access free API to kearn LLM using GPT-4 over extracted pdf texts

In [ ]:
#!pip install openai

In [ ]:
import openai
api_api_key= "sk-HtNeSf7N6mBCtkIRJdez2A"
client = openai.OpenAI(
    api_key= api_api_key, #"YOUR_API_KEY",
    base_url="https://llms.azurewebsites.net"
)

response = client.chat.completions.create(model="gpt-4",     #"MODEL_NAME (SEE TABLE)",
                                          messages = [
    {
        "role": "user",
        "content": "where isthe BCX Hackaton 2023 located?"
    }
])

print(response)

ChatCompletion(id='chatcmpl-8wuiWqMkioj1gdA6w79pzclQsUhAx', choices=[Choice(finish_reason='stop', index=0, logprobs=None, message=ChatCompletionMessage(content='As of now, the location for the BCX Hackaton 2023 is unknown as the organizers have not yet announced this information. Please monitor their official communications or website for the most recent updates.', role='assistant', function_call=None, tool_calls=None))], created=1709051736, model='gpt-4', object='chat.completion', system_fingerprint=None, usage=CompletionUsage(completion_tokens=40, prompt_tokens=19, total_tokens=59))


In [ ]:
text_by_page

{1: ' \nInstandsetzung Mechanik \nKRC KingDrive® Rollenförderer T \n \n \n \n \n \n',
 2: 'Kapitel VIII - Instandsetzung Mechanik \nKRC KingDrive® Rollenförderer T \nInhaltsverzeichnis \n \n1 \nKingDrive-Rolle, Befestigungssatz, ConnectorModule ......................................... 2 \n2 \nKingDrive-Bremsrolle, Befestigungssatz, ConnectorModule/Brake, \nVerbindungskabel CMB ....................................................................................... 8 \n3 \nSlave-Rolle, Rundriemen, Befestigungssatz Rolle 50 ........................................ 13 \n4 \nE-Bauteile........................................................................................................... 17 \n4.1 \nNetzteil, 48/24 V JunctionBox ...................................................................... 17 \n4.2 \nNetzteil, Bremslogikbox/Verteilbox............................................................... 20 \n4.3 \nPowerTrack, JumperCable, Halter PowerPlug ..............................

In [ ]:
import fitz  # PyMuPDF
import openai
import os

import openai
api_api_key= "sk-HtNeSf7N6mBCtkIRJdez2A"

# Step 1: Read and extract textual content from the given PDF
pdf_path = "/content/010_KRC T Instandsetzung Mechanik (DM0009AWQH)_V1.pdf"

def extract_text_from_pdf(pdf_path):
    text_by_page = {}
    with fitz.open(pdf_path) as doc:
        for page_num in range(len(doc)):
            page = doc.load_page(page_num)
            text = page.get_text()
            text_by_page[page_num + 1] = text
    return text_by_page

text_by_page = extract_text_from_pdf(pdf_path)

# Step 2: Apply LLM (GPT-3) for the input query prompt
openai.api_key = api_api_key, #"YOUR_API_KEY"  # Replace with your OpenAI API key

query = "Wie wechsele ich das Netzteil?"

def answer_question(question, context):
    response = openai.Completion.create(
        engine= "gpt-4", #"text-davinci-003",
        prompt=f"Question: {question}\nContext: {context}\nAnswer:",
        max_tokens=50
    )
    return response.choices[0].text.strip()

# Search for the query in the extracted text
answer = None
for page_num, text in text_by_page.items():
    if query.lower() in text.lower():
        answer = answer_question(query, text)
        print("Answer found in Page", page_num, ":", answer)
        break

if not answer:
    print("Answer not found in the provided PDF.")

# Step 3: Gather additional information from referenced PDF files (if available)
# For demonstration purposes, let's assume we have another PDF file in the same directory
additional_pdf_path = "/content/002_Sicherheitshinweis Instandsetzung (DM0009AU08)_V1.pdf"

if os.path.exists(additional_pdf_path):
    additional_text_by_page = extract_text_from_pdf(additional_pdf_path)
    for page_num, text in additional_text_by_page.items():
        if query.lower() in text.lower():
            additional_answer = answer_question(query, text)
            print("Additional information found in Page", page_num, ":", additional_answer)
            # You can continue processing additional references as needed
            break
    else:
        print("Additional information not found in the referenced PDF.")
else:
    print("No additional PDF file found.")


Answer not found in the provided PDF.
Additional information not found in the referenced PDF.


In [ ]:
text_by_page

{1: ' \nInstandsetzung Mechanik \nKRC KingDrive® Rollenförderer T \n \n \n \n \n \n',
 2: 'Kapitel VIII - Instandsetzung Mechanik \nKRC KingDrive® Rollenförderer T \nInhaltsverzeichnis \n \n1 \nKingDrive-Rolle, Befestigungssatz, ConnectorModule ......................................... 2 \n2 \nKingDrive-Bremsrolle, Befestigungssatz, ConnectorModule/Brake, \nVerbindungskabel CMB ....................................................................................... 8 \n3 \nSlave-Rolle, Rundriemen, Befestigungssatz Rolle 50 ........................................ 13 \n4 \nE-Bauteile........................................................................................................... 17 \n4.1 \nNetzteil, 48/24 V JunctionBox ...................................................................... 17 \n4.2 \nNetzteil, Bremslogikbox/Verteilbox............................................................... 20 \n4.3 \nPowerTrack, JumperCable, Halter PowerPlug ..............................

In [ ]:
import fitz  # PyMuPDF
import openai
import os

# Step 1: Read and extract textual content from the given PDFs
pdf1_path = "/content/010_KRC T Instandsetzung Mechanik (DM0009AWQH)_V1.pdf"
pdf2_path = "/content/002_Sicherheitshinweis Instandsetzung (DM0009AU08)_V1.pdf"

def extract_text_from_pdf(pdf_path):
    text_by_page = {}
    with fitz.open(pdf_path) as doc:
        for page_num in range(len(doc)):
            page = doc.load_page(page_num)
            text = page.get_text()
            text_by_page[page_num + 1] = text
    return text_by_page

text_by_page_pdf1 = extract_text_from_pdf(pdf1_path)
text_by_page_pdf2 = extract_text_from_pdf(pdf2_path)

# Step 2: Train LLM model using OpenAI API (if not already trained)
# Skip this step if the model is already trained
openai.api_key = api_api_key,

# Step 3: Return a generated answer for the input query prompt
def answer_question(question, context):
    response = openai.Completion.create(
        engine="gpt-4", #"text-davinci-003",
        prompt=f"Question: {question}\nContext: {context}\nAnswer:",
        max_tokens=50
    )
    return response.choices[0].text.strip()

query = "Wie wechsele ich das Netzteil?"

# Search for the query in the first PDF
answer_pdf1 = None
for page_num, text in text_by_page_pdf1.items():
    if query.lower() in text.lower():
        answer_pdf1 = answer_question(query, text)
        print("Answer found in PDF 1, Page", page_num, ":", answer_pdf1)
        break

if not answer_pdf1:
    print("Answer not found in the first PDF.")

# Step 4: Regenerate the answer if there are possible references in the second PDF
answer_pdf2 = None
for page_num, text in text_by_page_pdf2.items():
    if query.lower() in text.lower():
        answer_pdf2 = answer_question(query, text)
        print("Additional information found in PDF 2, Page", page_num, ":", answer_pdf2)
        # You can continue processing additional references as needed
        break

if not answer_pdf2:
    print("No additional information found in the second PDF.")


Answer not found in the first PDF.
No additional information found in the second PDF.


In [ ]:
import fitz  # PyMuPDF
import openai
import os

# Step 1: Read and extract textual content from the given PDF
pdf_path = "/content/010_KRC T Instandsetzung Mechanik (DM0009AWQH)_V1.pdf"

def extract_text_from_pdf(pdf_path):
    text_by_page = {}
    with fitz.open(pdf_path) as doc:
        for page_num in range(len(doc)):
            page = doc.load_page(page_num)
            text = page.get_text()
            text_by_page[page_num + 1] = text
    return text_by_page

text_by_page_pdf1 = extract_text_from_pdf(pdf_path)
all_text_pdf1 = '\n'.join(text_by_page_pdf1.values())

# Step 2: Train LLM model using OpenAI API
openai.api_key = api_api_key  # Replace with your OpenAI API key

response = openai.Completion.create(
    engine="gpt-4", #"text-davinci-003",
    prompt=all_text_pdf1,
    max_tokens=2000,
    n=1,
    stop=["###"]
)
trained_model_output = response.choices[0].text.strip()

# Step 3: Return a generated answer for the input query prompt
query = "Wie wechsele ich das Netzteil?"

def answer_question(question, context):
    response = openai.Completion.create(
        engine="text-davinci-003",
        prompt=f"Question: {question}\nContext: {context}\nAnswer:",
        max_tokens=50
    )
    return response.choices[0].text.strip()

answer = answer_question(query, trained_model_output)
print("Generated answer:", answer)

# Step 4: Regenerate the answer if there are possible references in the additional PDF
additional_pdf_path = "/content/002_Sicherheitshinweis Instandsetzung (DM0009AU08)_V1.pdf"

if os.path.exists(additional_pdf_path):
    text_by_page_pdf2 = extract_text_from_pdf(additional_pdf_path)
    all_text_pdf2 = '\n'.join(text_by_page_pdf2.values())

    response_additional = openai.Completion.create(
        engine="text-davinci-003",
        prompt=f"Question: {query}\nContext: {all_text_pdf2}\nAnswer:",
        max_tokens=50
    )
    additional_answer = response_additional.choices[0].text.strip()
    print("Additional information found:", additional_answer)
else:
    print("No additional PDF file found.")


APIRemovedInV1: 

You tried to access openai.Completion, but this is no longer supported in openai>=1.0.0 - see the README at https://github.com/openai/openai-python for the API.

You can run `openai migrate` to automatically upgrade your codebase to use the 1.0.0 interface. 

Alternatively, you can pin your installation to the old version, e.g. `pip install openai==0.28`

A detailed migration guide is available here: https://github.com/openai/openai-python/discussions/742


In [ ]:
from transformers import GPT2Tokenizer, GPT2LMHeadModel, TextDataset, DataCollatorForLanguageModeling, Trainer, TrainingArguments

# Step 1: Define your training text data
#training_text = [
#    "Your first piece of training text here.",
#    "Your second piece of training text here.",
#    # Add more training text as needed
#]

training_text= all_text_pdf1

# Step 2: Tokenize the training text
tokenizer = GPT2Tokenizer.from_pretrained("gpt2")
train_encodings = tokenizer(training_text, return_tensors="pt", padding=True, truncation=True)

# Step 3: Create a dataset from the tokenized encodings
train_dataset = TextDataset(train_encodings)

# Step 4: Define the LM model to be trained
model = GPT2LMHeadModel.from_pretrained("gpt2")

# Step 5: Define training arguments
training_args = TrainingArguments(
    output_dir="./output",
    overwrite_output_dir=True,
    num_train_epochs=3,
    per_device_train_batch_size=4,
    save_steps=10_000,
    save_total_limit=2,
    prediction_loss_only=True,
)

# Step 6: Define the Trainer object
trainer = Trainer(
    model=model,
    args=training_args,
    data_collator=DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False),
    train_dataset=train_dataset,
)

# Step 7: Train the LM on the provided training data
trainer.train()

# Step 8: Generate an answer for the prompt "Wie wechsele ich das Netzteil?"
prompt = "Wie wechsele ich das Netzteil?"
generated_output = trainer.model.generate(tokenizer(prompt, return_tensors="pt")["input_ids"], max_length=50)
generated_text = tokenizer.decode(generated_output[0], skip_special_tokens=True)
print("Generated answer:", generated_text)


ValueError: Asking to pad but the tokenizer does not have a padding token. Please select a token to use as `pad_token` `(tokenizer.pad_token = tokenizer.eos_token e.g.)` or add a new pad token via `tokenizer.add_special_tokens({'pad_token': '[PAD]'})`.

# Apply LLM over extracted technical PDF texts

In [ ]:
#!pip install transformers[torch]
#!pip install accelerate -U
#! pip install -U accelerate
#! pip install -U transformers

In [ ]:
from transformers import GPT2Tokenizer, GPT2LMHeadModel, TextDataset, DataCollatorForLanguageModeling, Trainer, TrainingArguments
import fitz  # PyMuPDF
import openai
import os

# Step 1: Read and extract textual content from the given PDFs
pdf1_path = "/content/010_KRC T Instandsetzung Mechanik (DM0009AWQH)_V1.pdf"
pdf2_path = "/content/002_Sicherheitshinweis Instandsetzung (DM0009AU08)_V1.pdf"

def extract_text_from_pdf(pdf_path):
    text_by_page = {}
    with fitz.open(pdf_path) as doc:
        for page_num in range(len(doc)):
            page = doc.load_page(page_num)
            text = page.get_text()
            text_by_page[page_num + 1] = text
    return text_by_page

text_by_page_pdf1 = extract_text_from_pdf(pdf1_path)
text_by_page_pdf2 = extract_text_from_pdf(pdf2_path)

training_text = text_by_page_pdf1
# Step 1: Define your training text data
#training_text = [
#    "Your first piece of training text here.",
#    "Your second piece of training text here.",
#    # Add more training text as needed
#]

# Step 2: Tokenize the training text
tokenizer = GPT2Tokenizer.from_pretrained("gpt2")
tokenizer.pad_token = tokenizer.eos_token  # Set padding token to end-of-sequence token

# Step 3: Create a dataset from the tokenized encodings
train_dataset = TextDataset(
    tokenizer=tokenizer,
    text=training_text,
    block_size=128  # Adjust block size as needed
)

# Step 4: Define the LM model to be trained
model = GPT2LMHeadModel.from_pretrained("gpt2")

# Step 5: Define training arguments
training_args = TrainingArguments(
    output_dir="./output",
    overwrite_output_dir=True,
    num_train_epochs=3,
    per_device_train_batch_size=4,
    save_steps=10_000,
    save_total_limit=2,
    prediction_loss_only=True,
)

# Step 6: Define the Trainer object
trainer = Trainer(
    model=model,
    args=training_args,
    data_collator=DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False),
    train_dataset=train_dataset,
)

# Step 7: Train the LM on the provided training data
trainer.train()

# Step 8: Generate an answer for the prompt "Wie wechsele ich das Netzteil?"
prompt = "Wie wechsele ich das Netzteil?"
generated_output = trainer.model.generate(tokenizer(prompt, return_tensors="pt")["input_ids"], max_length=50)
generated_text = tokenizer.decode(generated_output[0], skip_special_tokens=True)
print("Generated answer:", generated_text)


/usr/local/lib/python3.10/dist-packages/huggingface_hub/utils/_token.py:88: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


TypeError: TextDataset.__init__() got an unexpected keyword argument 'text'

In [ ]:
import torch
from transformers import GPT2Tokenizer, GPT2LMHeadModel, TextDataset, DataCollatorForLanguageModeling, Trainer, TrainingArguments

import fitz  # PyMuPDF
import openai
import os

# Step 1: Read and extract textual content from the given PDFs
pdf1_path = "/content/010_KRC T Instandsetzung Mechanik (DM0009AWQH)_V1.pdf"
pdf2_path = "/content/002_Sicherheitshinweis Instandsetzung (DM0009AU08)_V1.pdf"

def extract_text_from_pdf(pdf_path):
    text_by_page = {}
    with fitz.open(pdf_path) as doc:
        for page_num in range(len(doc)):
            page = doc.load_page(page_num)
            text = page.get_text()
            text_by_page[page_num + 1] = text
    return text_by_page

text_by_page_pdf1 = extract_text_from_pdf(pdf1_path)
text_by_page_pdf2 = extract_text_from_pdf(pdf2_path)

training_text = text_by_page_pdf1
# Step 1: Define your training text data
#training_text = [
#    "Your first piece of training text here.",
#    "Your second piece of training text here.",
#    # Add more training text as needed
#]

# Step 2: Tokenize the training text
tokenizer = GPT2Tokenizer.from_pretrained("gpt2")
tokenizer.pad_token = tokenizer.eos_token  # Set padding token to end-of-sequence token

# Step 3: Create a PyTorch Dataset from the tokenized text
train_dataset = TextDataset(
    tokenizer=tokenizer,
    file_path=None,  # No file path needed since we're passing text directly
    block_size=128,  # Adjust block size as needed
    overwrite_cache=False,  # Set to True if you want to overwrite cache
    text=training_text  # Pass the training text directly
)

# Step 4: Define the LM model to be trained
model = GPT2LMHeadModel.from_pretrained("gpt2")

# Step 5: Define training arguments
training_args = TrainingArguments(
    output_dir="./output",
    overwrite_output_dir=True,
    num_train_epochs=3,
    per_device_train_batch_size=4,
    save_steps=10_000,
    save_total_limit=2,
    prediction_loss_only=True,
)

# Step 6: Define the Trainer object
trainer = Trainer(
    model=model,
    args=training_args,
    data_collator=DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False),
    train_dataset=train_dataset,
)

# Step 7: Train the LM on the provided training data
trainer.train()

# Step 8: Generate an answer for the prompt "Wie wechsele ich das Netzteil?"
prompt = "Wie wechsele ich das Netzteil?"
generated_output = trainer.model.generate(tokenizer(prompt, return_tensors="pt")["input_ids"], max_length=50)
generated_text = tokenizer.decode(generated_output[0], skip_special_tokens=True)
print("Generated answer:", generated_text)


TypeError: TextDataset.__init__() got an unexpected keyword argument 'text'